# Diagnosing SHAP attributions on a benchmark airline satisfaction survey

Revised analysis for the IEEE Access resubmission. Run all cells in order (no GPU needed; seeds fixed at 42). Results are written to `results/` and figures to `figures/`. Zero ratings are coded as missing because the data dictionary defines 0 as "not applicable". Binary variables are single 0/1 columns. The slowest steps (interventional SHAP, interaction values, repeated CV) cache their results.

## 1. Data, models and out-of-fold SHAP values

In [ ]:
# Cell 1: Imports and settings
import os, sys, time, warnings, pickle
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
import shap
from scipy import stats
from scipy.stats import friedmanchisquare
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score)

warnings.filterwarnings('ignore')
SEED = 42
N_FOLDS = 5
DATA_PATH = 'airline_passenger_satisfaction.csv'
OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)



In [ ]:
# Cell 2: Load data; 0 ratings are set to missing (they lie outside the 1-5 scale)
df = pd.read_csv(DATA_PATH)
assert df.shape == (129880, 24)
service_cols = ['Departure and Arrival Time Convenience', 'Ease of Online Booking', 'Check-in Service',
                'Online Boarding', 'Gate Location', 'On-board Service', 'Seat Comfort', 'Leg Room Service',
                'Cleanliness', 'Food and Drink', 'In-flight Service', 'In-flight Wifi Service',
                'In-flight Entertainment', 'Baggage Handling']
raw = df.copy()
df[service_cols] = df[service_cols].replace(0, np.nan)
y = (df['Satisfaction'] == 'Satisfied').astype(int).values



In [ ]:
# Cell 3: Encoding - one 0/1 column per binary variable (revision), three dummies for the 3-level class
X = df[['Age', 'Flight Distance', 'Departure Delay', 'Arrival Delay'] + service_cols].copy()
X['Male'] = (df['Gender'] == 'Male').astype(float)
X['First-time customer'] = (df['Customer Type'] == 'First-time').astype(float)
X['Business travel'] = (df['Type of Travel'] == 'Business').astype(float)
for cl in ['Business', 'Economy', 'Economy Plus']:
    X[f'Class {cl}'] = (df['Class'] == cl).astype(float)
X = X.astype('float32')
feature_names = X.columns.tolist()
num_cols = ['Age', 'Flight Distance', 'Departure Delay', 'Arrival Delay'] + service_cols
print('Features:', len(feature_names))



In [ ]:
# Cell 4: Candidate configurations
# XGBoost and LightGBM lists are drawn exactly as in the first version (same RNG sequence), so they are unchanged.
rng = np.random.RandomState(SEED)
_old_rf = [{'max_depth': [None, 20, 30][rng.randint(3)], 'min_samples_leaf': int(rng.choice([1, 2, 5])),
            'max_features': ['sqrt', 0.3][rng.randint(2)]} for _ in range(4)]
xgb_configs = [{'max_depth': int(rng.choice([4, 6, 8, 10])), 'learning_rate': float(rng.choice([0.03, 0.05, 0.1])),
                'subsample': float(rng.choice([0.7, 0.8, 1.0])), 'colsample_bytree': float(rng.choice([0.6, 0.8, 1.0])),
                'min_child_weight': int(rng.choice([1, 3, 5])), 'reg_lambda': float(rng.choice([0.5, 1.0, 5.0]))}
               for _ in range(10)]
lgb_configs = [{'num_leaves': int(rng.choice([31, 63, 127, 255])), 'learning_rate': float(rng.choice([0.03, 0.05, 0.1])),
                'min_child_samples': int(rng.choice([10, 20, 50])), 'colsample_bytree': float(rng.choice([0.6, 0.8, 1.0])),
                'subsample': float(rng.choice([0.7, 0.8, 1.0])), 'reg_lambda': float(rng.choice([0.0, 1.0, 5.0]))}
               for _ in range(10)]
# Revision: wider logistic-regression grid and 10 DISTINCT random-forest configurations
lr_grid = [1e-3, 1e-2, 1e-1, 1, 10, 100, 1000]
rf_space = [{'max_depth': d, 'min_samples_leaf': l, 'max_features': f}
            for d in [None, 15, 20, 30] for l in [1, 2, 5, 10] for f in ['sqrt', 0.3, 0.5]]
rf_pick = np.random.RandomState(SEED + 1).choice(len(rf_space), 10, replace=False)
rf_configs = [rf_space[i] for i in rf_pick]
pd.DataFrame(rf_configs).to_csv(f'{OUT_DIR}/search_space_rf.csv', index_label='config')
pd.DataFrame(xgb_configs).to_csv(f'{OUT_DIR}/search_space_xgboost.csv', index_label='config')
pd.DataFrame(lgb_configs).to_csv(f'{OUT_DIR}/search_space_lightgbm.csv', index_label='config')
print(pd.DataFrame(rf_configs).to_string())



In [ ]:
# Cell 5: Folds; all learned preprocessing inside the training part
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = []
for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_te = X.iloc[tr_idx].copy(), X.iloc[te_idx].copy()
    y_tr, y_te = y[tr_idx], y[te_idx]
    X_in, X_val, y_in, y_val = train_test_split(X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED)
    med = X_in[num_cols].median()
    imp = {}
    for name, part in [('in', X_in), ('val', X_val), ('te', X_te)]:
        p = part.copy()
        for c in service_cols + ['Arrival Delay']:
            p[c + ' (missing)'] = p[c].isna().astype('float32')
        p[num_cols] = p[num_cols].fillna(med)
        imp[name] = p
    scaler = StandardScaler().fit(imp['in'])
    folds.append({'te_idx': te_idx, 'y_in': y_in, 'y_val': y_val, 'y_te': y_te,
                  'raw_in': X_in, 'raw_val': X_val, 'raw_te': X_te,
                  'imp_in': imp['in'], 'imp_val': imp['val'], 'imp_te': imp['te'],
                  'sc_in': scaler.transform(imp['in']), 'sc_val': scaler.transform(imp['val']),
                  'sc_te': scaler.transform(imp['te'])})
print('Imputed feature count (LR/RF):', folds[0]['imp_in'].shape[1])



In [ ]:
# Cell 6: Train and evaluate the four models
model_names = ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']
results_rows, chosen, best_models = [], [], {}
oof_proba = {m: np.full(len(y), np.nan) for m in model_names}

def record(m, s, f, best_p, best_cfg, best_auc, best_params):
    oof_proba[m][f['te_idx']] = best_p
    pred = (best_p >= 0.5).astype(int)
    results_rows.append({'Model': m, 'Fold': s + 1, 'Accuracy': accuracy_score(f['y_te'], pred),
                         'Precision': precision_score(f['y_te'], pred), 'Recall': recall_score(f['y_te'], pred),
                         'F1': f1_score(f['y_te'], pred), 'ROC-AUC': roc_auc_score(f['y_te'], best_p),
                         'PR-AUC': average_precision_score(f['y_te'], best_p), 'Chosen config': best_cfg,
                         'Validation ROC-AUC': best_auc})
    chosen.append({'Model': m, 'Fold': s + 1, 'Validation ROC-AUC': round(best_auc, 4), **best_params})
    print(f"{m} fold {s+1}: cfg={best_cfg} valAUC={best_auc:.4f} acc={accuracy_score(f['y_te'], pred):.4f}", flush=True)

def cached(m):
    p = f'{OUT_DIR}/cache_{m.replace(" ", "_")}.pkl'
    if os.path.exists(p):
        d = pickle.load(open(p, 'rb'))
        results_rows.extend(d['rows']); chosen.extend(d['chosen']); oof_proba[m][:] = d['oof']
        if m == 'LightGBM': best_models.update(d['models'])
        print('loaded cached', m); return True
    return False
def save_cache(m):
    pickle.dump({'rows': [r for r in results_rows if r['Model'] == m], 'chosen': [c for c in chosen if c['Model'] == m],
                 'oof': oof_proba[m], 'models': best_models if m == 'LightGBM' else None},
                open(f'{OUT_DIR}/cache_{m.replace(" ", "_")}.pkl', 'wb'))

for s, f in enumerate(folds if not (cached('Logistic Regression')) else []):
    best_auc = -1
    for ci, C in enumerate(lr_grid):
        model = LogisticRegression(C=C, max_iter=3000, random_state=SEED).fit(f['sc_in'], f['y_in'])
        a = roc_auc_score(f['y_val'], model.predict_proba(f['sc_val'])[:, 1])
        if a > best_auc:
            best_auc, best_cfg, best_params, best_p = a, ci, {'C': C}, model.predict_proba(f['sc_te'])[:, 1]
    record('Logistic Regression', s, f, best_p, best_cfg, best_auc, best_params)
    if s == N_FOLDS - 1: save_cache('Logistic Regression')

for s, f in enumerate(folds if not (cached('Random Forest')) else []):
    best_auc = -1
    for ci, c in enumerate(rf_configs):
        model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED, **c).fit(f['imp_in'], f['y_in'])
        a = roc_auc_score(f['y_val'], model.predict_proba(f['imp_val'])[:, 1])
        if a > best_auc:
            best_auc, best_cfg, best_params, best_p = a, ci, dict(c), model.predict_proba(f['imp_te'])[:, 1]
        del model
    record('Random Forest', s, f, best_p, best_cfg, best_auc, best_params)
    if s == N_FOLDS - 1: save_cache('Random Forest')

xgb_iters = []
for s, f in enumerate(folds if not (cached('XGBoost')) else []):
    best_auc = -1
    for ci, c in enumerate(xgb_configs):
        model = xgb.XGBClassifier(n_estimators=2000, eval_metric='logloss', early_stopping_rounds=50,
                                  tree_method='hist', random_state=SEED, n_jobs=-1, **c)
        model.fit(f['raw_in'], f['y_in'], eval_set=[(f['raw_val'], f['y_val'])], verbose=False)
        a = roc_auc_score(f['y_val'], model.predict_proba(f['raw_val'])[:, 1])
        if a > best_auc:
            best_auc, best_cfg, best_params, best_p = a, ci, dict(c), model.predict_proba(f['raw_te'])[:, 1]
    record('XGBoost', s, f, best_p, best_cfg, best_auc, best_params)
    if s == N_FOLDS - 1: save_cache('XGBoost')

for s, f in enumerate(folds if not cached('LightGBM') else []):
    best_auc = -1
    for ci, c in enumerate(lgb_configs):
        model = lgb.LGBMClassifier(n_estimators=2000, subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1, **c)
        model.fit(f['raw_in'], f['y_in'], eval_set=[(f['raw_val'], f['y_val'])], eval_metric='binary_logloss',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        a = roc_auc_score(f['y_val'], model.predict_proba(f['raw_val'])[:, 1])
        if a > best_auc:
            best_auc, best_cfg, best_params, best_p = a, ci, dict(c), model.predict_proba(f['raw_te'])[:, 1]
            best_models[s] = model
    record('LightGBM', s, f, best_p, best_cfg, best_auc, best_params)
    if s == N_FOLDS - 1: save_cache('LightGBM')



In [ ]:
# Cell 7: Results table, chosen configurations, tests (McNemar/Friedman now reported in the supplement)
results_df = pd.DataFrame(results_rows)
results_df.to_csv(f'{OUT_DIR}/fold_results.csv', index=False)
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']
model_names = [m for m in model_names if m in set(results_df.Model)]
mean_df = results_df.groupby('Model')[metrics].mean().loc[model_names]
std_df = results_df.groupby('Model')[metrics].std().loc[model_names]
main_table = mean_df.round(4).astype(str) + ' ± ' + std_df.round(4).astype(str)
main_table.to_csv(f'{OUT_DIR}/table_main_results.csv')
print(main_table.to_string())
chosen_df = pd.DataFrame(chosen); chosen_df.to_csv(f'{OUT_DIR}/table_chosen_hyperparameters.csv', index=False)
print(chosen_df.to_string())
print('LightGBM stopping iterations:', [int(best_models[s].best_iteration_) for s in range(N_FOLDS)])

if len(model_names) == 4:
  f1_matrix = results_df.pivot(index='Fold', columns='Model', values='F1')[model_names]
  fr_stat, fr_p = friedmanchisquare(*[f1_matrix[c] for c in model_names])
  c1 = (oof_proba['LightGBM'] >= 0.5) == y; c2 = (oof_proba['XGBoost'] >= 0.5) == y
  mc_table = [[int(np.sum(c1 & c2)), int(np.sum(c1 & ~c2))], [int(np.sum(~c1 & c2)), int(np.sum(~c1 & ~c2))]]
  mc = mcnemar(mc_table, exact=False, correction=True)
  n_te = len(folds[0]['y_te']); n_tr = len(folds[0]['y_in']) + len(folds[0]['y_val'])
  tt_rows = []
  for met in metrics:
      a = results_df[results_df.Model == 'LightGBM'].sort_values('Fold')[met].values
      b = results_df[results_df.Model == 'XGBoost'].sort_values('Fold')[met].values
      d = a - b; k = len(d)
      t = d.mean() / np.sqrt((1 / k + n_te / n_tr) * d.var(ddof=1)) if d.var(ddof=1) > 0 else np.nan
      p = 2 * stats.t.sf(abs(t), k - 1)
      tt_rows.append({'Metric': met, 'Mean difference': d.mean(), 't': t, 'p': p})
  tt = pd.DataFrame(tt_rows); tt.to_csv(f'{OUT_DIR}/table_corrected_ttest.csv', index=False)
  print(tt.round(4).to_string())
  pd.DataFrame([{'Test': 'Friedman (F1, 4 models)', 'Statistic': fr_stat, 'p-value': fr_p},
                {'Test': 'McNemar (LightGBM vs XGBoost)', 'Statistic': mc.statistic, 'p-value': mc.pvalue,
                 'Detail': f'only LightGBM correct: {mc_table[0][1]}; only XGBoost correct: {mc_table[1][0]}'}]
               ).to_csv(f'{OUT_DIR}/table_statistical_tests_supplement.csv', index=False)
  print('Friedman', fr_stat, fr_p, '| McNemar', mc.statistic, mc.pvalue, mc_table)



In [ ]:
# Cell 8: Out-of-fold path-dependent TreeSHAP (log-odds), 3,000 test passengers per fold
N_SHAP_PER_FOLD = 3000
rs = np.random.RandomState(SEED)
shap_parts, idx_parts, pick_parts = [], [], []
for s, f in enumerate(folds):
    pick = rs.choice(len(f['te_idx']), N_SHAP_PER_FOLD, replace=False)
    sv = shap.TreeExplainer(best_models[s]).shap_values(f['raw_te'].iloc[pick])
    sv = np.asarray(sv[1] if isinstance(sv, list) else sv)
    if sv.ndim == 3: sv = sv[:, :, 1]
    shap_parts.append(sv.astype('float32')); idx_parts.append(f['te_idx'][pick]); pick_parts.append(pick)
shap_idx = np.concatenate(idx_parts); shap_all = np.vstack(shap_parts)
fold_of = np.repeat(np.arange(N_FOLDS), N_SHAP_PER_FOLD)
np.save(f'{OUT_DIR}/shap_values_oof.npy', shap_all); np.save(f'{OUT_DIR}/shap_passenger_index.npy', shap_idx)
mean_abs = pd.Series(np.abs(shap_all).mean(axis=0), index=feature_names).sort_values(ascending=False)
mean_abs.to_csv(f'{OUT_DIR}/shap_mean_abs.csv'); print(mean_abs.round(3).to_string())



In [ ]:
# Cell 9: Save state for the analysis cells
with open(f'{OUT_DIR}/state.pkl', 'wb') as fh:
    pickle.dump({'feature_names': feature_names, 'service_cols': service_cols, 'best_models': best_models,
                 'folds_idx': [(f['te_idx']) for f in folds], 'shap_idx': shap_idx, 'fold_of': fold_of,
                 'pick_parts': pick_parts, 'lgb_configs': lgb_configs, 'results_df': results_df,
                 'oof_proba': oof_proba}, fh)
print('done')


## 2. Rating-level profiles, segment comparisons and diagnostic checks

In [ ]:
# Cell B0: Setup - reload data, encoding and the fitted fold models from part A
import os, time, pickle, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

warnings.filterwarnings('ignore')
SEED = 42; N_FOLDS = 5
DATA_PATH = 'airline_passenger_satisfaction.csv'; OUT_DIR = 'results'
st = pickle.load(open(f'{OUT_DIR}/state.pkl', 'rb'))
feature_names, service_cols, best_models = st['feature_names'], st['service_cols'], st['best_models']
shap_idx, fold_of, pick_parts, lgb_configs, results_df = st['shap_idx'], st['fold_of'], st['pick_parts'], st['lgb_configs'], st['results_df']
raw = pd.read_csv(DATA_PATH)
df = raw.copy(); df[service_cols] = df[service_cols].replace(0, np.nan)
y = (df['Satisfaction'] == 'Satisfied').astype(int).values
X = df[['Age', 'Flight Distance', 'Departure Delay', 'Arrival Delay'] + service_cols].copy()
X['Male'] = (df['Gender'] == 'Male').astype(float)
X['First-time customer'] = (df['Customer Type'] == 'First-time').astype(float)
X['Business travel'] = (df['Type of Travel'] == 'Business').astype(float)
for cl in ['Business', 'Economy', 'Economy Plus']:
    X[f'Class {cl}'] = (df['Class'] == cl).astype(float)
X = X.astype('float32'); assert X.columns.tolist() == feature_names
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(X, y))
shap_all = np.load(f'{OUT_DIR}/shap_values_oof.npy')
R = df[service_cols].iloc[shap_idx].reset_index(drop=True)            # ratings of explained passengers (NaN = 0)
seg = raw[['Type of Travel', 'Customer Type', 'Class', 'Flight Distance']].iloc[shap_idx].reset_index(drop=True)
TT = seg['Type of Travel'].values; CT = seg['Customer Type'].values; CL = seg['Class'].values
dist_t = pd.qcut(raw['Flight Distance'], 3, labels=['short', 'medium', 'long']).iloc[shap_idx].reset_index(drop=True).values
KEY = ['In-flight Wifi Service', 'Online Boarding']
B = 500
fidx = {c: feature_names.index(c) for c in feature_names}
print('Explained passengers:', len(shap_idx), '| business', (TT == 'Business').sum(), '| personal', (TT == 'Personal').sum(),
      '| returning', (CT == 'Returning').sum(), '| first-time', (CT == 'First-time').sum())

def contrast(v, s, m, w_low=None, w_high=None):
    """Top-rating contrast: mean attribution at 4-5 minus mean at 1-3 among rows in mask m.
    With w_low/w_high (dicts rating->weight), the bins are reweighted to a common rating distribution."""
    if w_low is None:
        top, low = m & (v >= 4), m & (v <= 3)
        if not top.any() or not low.any(): return np.nan
        return s[top].mean() - s[low].mean()
    def wmean(ws):
        num = den = 0.0
        for r, w in ws.items():
            k = m & (v == r)
            if k.any(): num += w * s[k].mean(); den += w
        return num / den if den > 0 else np.nan
    return wmean(w_high) - wmean(w_low)

def boot_diff(fun, n, rs, B=B):
    out = np.empty(B)
    for b in range(B):
        out[b] = fun(rs.randint(0, n, n))
    return np.nanpercentile(out, [2.5, 97.5]), out



In [ ]:
# Cell B1: Descriptive tables - Table I with both outcome columns, satisfaction by rating with counts, zero-rating profile
t1 = raw.assign(sat=y).groupby(['Type of Travel', 'Class'])['sat'].agg(['size', 'mean'])
t1['Satisfied (%)'] = (100 * t1['mean']).round(1); t1['Neutral or dissatisfied (%)'] = (100 - 100 * t1['mean']).round(1)
t1 = t1.drop(columns='mean').rename(columns={'size': 'Passengers'}); t1.to_csv(f'{OUT_DIR}/table1_segments.csv'); print(t1)
rows = []
for c in service_cols:
    row = {'Service': c}
    for r in range(6):
        m = (raw[c] == r).values; row[f'n {r}'] = int(m.sum()); row[f'% sat {r}'] = round(100 * y[m].mean(), 1) if m.any() else np.nan
    rows.append(row)
t4 = pd.DataFrame(rows); t4.to_csv(f'{OUT_DIR}/table_satisfaction_by_rating.csv', index=False)
print(t4[['Service'] + [f'% sat {r}' for r in range(1, 6)] + [f'n {r}' for r in range(1, 6)]].to_string())
wifi_tab = pd.crosstab(raw['Type of Travel'], raw['In-flight Wifi Service'], values=y, aggfunc='mean').mul(100).round(1)
wifi_n = pd.crosstab(raw['Type of Travel'], raw['In-flight Wifi Service'])
wifi_tab.to_csv(f'{OUT_DIR}/table_wifi_by_travel.csv'); wifi_n.to_csv(f'{OUT_DIR}/table_wifi_by_travel_counts.csv')
print(wifi_tab); print(wifi_n)
# Zero ratings: by segment, class and service; satisfaction among zero-raters by segment; co-occurrence
Z = (raw[service_cols] == 0)
zr = []
for (tt, cl), g in raw.groupby(['Type of Travel', 'Class']):
    zg = Z.loc[g.index]
    zr.append({'Type of Travel': tt, 'Class': cl, 'Passengers': len(g), 'Any zero (%)': round(100 * zg.any(axis=1).mean(), 1),
               **{f'Zero {c} (%)': round(100 * zg[c].mean(), 2) for c in ['In-flight Wifi Service', 'Ease of Online Booking',
                  'Departure and Arrival Time Convenience', 'Online Boarding']},
               'Satisfied | zero Wi-Fi (%)': round(100 * y[g.index[zg['In-flight Wifi Service']]].mean(), 1) if zg['In-flight Wifi Service'].any() else np.nan,
               'n zero Wi-Fi': int(zg['In-flight Wifi Service'].sum()),
               'Satisfied | no zero (%)': round(100 * y[g.index[~zg.any(axis=1)]].mean(), 1)})
zr = pd.DataFrame(zr); zr.to_csv(f'{OUT_DIR}/table_zero_profile_segment.csv', index=False); print(zr.to_string())
zc = pd.DataFrame({'Zero count': Z.sum(), 'Satisfied given zero (%)': [round(100 * y[Z[c].values].mean(), 1) if Z[c].any() else np.nan for c in service_cols]})
zc.to_csv(f'{OUT_DIR}/table_zero_by_service.csv'); print(zc)
nz_per = Z.sum(axis=1); print('Number of zero ratings per passenger:', nz_per.value_counts().sort_index().to_dict())
co = Z[['In-flight Wifi Service', 'Ease of Online Booking', 'Online Boarding', 'Departure and Arrival Time Convenience']]
print('Co-occurrence among passengers with zero Wi-Fi:', (co[Z['In-flight Wifi Service']].mean() * 100).round(1).to_dict())
zw = Z['In-flight Wifi Service'].values
print('Zero Wi-Fi: mean of other ratings vs others:', raw.loc[zw, service_cols].replace(0, np.nan).mean().mean().round(2),
      raw.loc[~zw, service_cols].replace(0, np.nan).mean().mean().round(2))
pd.DataFrame({'zero Wi-Fi': raw.loc[zw, service_cols].replace(0, np.nan).mean(), 'other': raw.loc[~zw, service_cols].replace(0, np.nan).mean()}).round(2).to_csv(f'{OUT_DIR}/table_zero_wifi_other_ratings.csv')
print('Zero Wi-Fi share by travel type:', raw.groupby('Type of Travel')['In-flight Wifi Service'].apply(lambda s: round(100 * (s == 0).mean(), 2)).to_dict())
print('Zero Wi-Fi share by customer type:', raw.groupby('Customer Type')['In-flight Wifi Service'].apply(lambda s: round(100 * (s == 0).mean(), 2)).to_dict())



In [ ]:
# Cell B2: Main (path-dependent, log-odds) summaries - grouped importance, rating profiles, contrasts, segment differences
imp = pd.Series(np.abs(shap_all).mean(axis=0), index=feature_names)
cls_cols = [fidx[c] for c in feature_names if c.startswith('Class ')]
imp_g = imp.drop([c for c in feature_names if c.startswith('Class ')])
imp_g['Class (3 columns)'] = np.abs(shap_all[:, cls_cols].sum(axis=1)).mean()
imp_g = imp_g.sort_values(ascending=False); imp_g.to_csv(f'{OUT_DIR}/shap_mean_abs_grouped.csv'); print(imp_g.round(3).to_string())
seg_defs = [('All', np.ones(len(R), bool)), ('Business travel', TT == 'Business'), ('Personal travel', TT == 'Personal'),
            ('Returning customers', CT == 'Returning'), ('First-time customers', CT == 'First-time')]
lv = []
for c in service_cols:
    j = fidx[c]; v = R[c].values
    for sn, m in seg_defs[:3]:
        for r in range(1, 6):
            k = m & (v == r)
            lv.append({'Attribute': c, 'Segment': sn, 'Rating': r, 'Passengers': int(k.sum()), 'Mean SHAP': shap_all[k, j].mean() if k.any() else np.nan})
lv = pd.DataFrame(lv); lv.to_csv(f'{OUT_DIR}/table_shap_by_rating.csv', index=False)
print(lv[lv.Segment == 'All'].pivot(index='Attribute', columns='Rating', values='Mean SHAP').round(2).to_string())
rs = np.random.RandomState(SEED); te_rows = []
for c in service_cols:
    j = fidx[c]; v = R[c].values; s = shap_all[:, j]
    for sn, m in seg_defs:
        ii = np.where(m & ~np.isnan(v))[0]; vv, ss = v[ii], s[ii]; one = np.ones(len(ii), bool)
        est = contrast(vv, ss, one)
        (lo, hi), _ = boot_diff(lambda b: contrast(vv[b], ss[b], one), len(ii), rs)
        te_rows.append({'Attribute': c, 'Segment': sn, 'n': len(ii), 'n 4-5': int((vv >= 4).sum()), 'Contrast': est, 'CI low': lo, 'CI high': hi})
te_tab = pd.DataFrame(te_rows); te_tab.to_csv(f'{OUT_DIR}/table_top_rating_effects.csv', index=False)
print(te_tab.pivot(index='Attribute', columns='Segment', values='Contrast').round(2).to_string())

def seg_diff_table(S, label, rs, ratings=R, tt=TT, ct=CT, weights=None):
    out = []
    for c in service_cols:
        j = fidx[c]; v = ratings[c].values; s = S[:, j]; ok = ~np.isnan(v)
        for a, b_, g in [('Personal', 'Business', tt), ('First-time', 'Returning', ct)]:
            ii = np.where(ok)[0]
            f = lambda bi: contrast(v[bi], s[bi], g[bi] == a) - contrast(v[bi], s[bi], g[bi] == b_)
            est = f(ii); (lo, hi), _ = boot_diff(lambda b: f(ii[b]), len(ii), rs)
            out.append({'Analysis': label, 'Attribute': c, 'Comparison': f'{a} minus {b_.lower()}', 'Difference': est, 'CI low': lo, 'CI high': hi})
    return pd.DataFrame(out)
sd_main = seg_diff_table(shap_all, 'Main (path-dependent, log-odds)', np.random.RandomState(SEED))
sd_main.to_csv(f'{OUT_DIR}/table_segment_differences.csv', index=False); print(sd_main.round(3).to_string())
# adjacent-rating contrasts (2,000 resamples)
rs = np.random.RandomState(SEED); adj = []
for sn, m in seg_defs[:3]:
    for c in service_cols:
        j = fidx[c]; ii = np.where(m & R[c].notna().values)[0]; vv, ss = R[c].values[ii], shap_all[ii, j]
        est = np.diff([ss[vv == r].mean() for r in range(1, 6)]); bt = np.empty((2000, 4))
        for b in range(2000):
            bi = rs.randint(0, len(ii), len(ii)); bt[b] = np.diff([ss[bi][vv[bi] == r].mean() for r in range(1, 6)])
        lo, hi = np.percentile(bt, [2.5, 97.5], axis=0)
        for k, step in enumerate(['1->2', '2->3', '3->4', '4->5']):
            adj.append({'Segment': sn, 'Attribute': c, 'Step': step, 'Contrast': est[k], 'CI low': lo[k], 'CI high': hi[k]})
adj = pd.DataFrame(adj); adj.to_csv(f'{OUT_DIR}/table_adjacent_contrasts.csv', index=False)
print(adj[(adj.Segment == 'All') & adj.Attribute.isin(KEY)].round(3).to_string())



In [ ]:
# Cell B3: Composition within rating bins - per-rating segment comparison and reweighting to a common rating distribution
per_r = []; rs = np.random.RandomState(SEED)
for c in service_cols:
    j = fidx[c]; v = R[c].values; s = shap_all[:, j]
    for r in range(1, 6):
        mP, mB = (v == r) & (TT == 'Personal'), (v == r) & (TT == 'Business')
        nP, nB = int(mP.sum()), int(mB.sum())
        est = s[mP].mean() - s[mB].mean() if nP and nB else np.nan
        ii = np.where(v == r)[0]; tt_i = TT[ii]; s_i = s[ii]
        (lo, hi), _ = boot_diff(lambda b: s_i[b][tt_i[b] == 'Personal'].mean() - s_i[b][tt_i[b] == 'Business'].mean(), len(ii), rs, B=500)
        per_r.append({'Attribute': c, 'Rating': r, 'n business': nB, 'n personal': nP, 'Mean SHAP business': s[mB].mean() if nB else np.nan,
                      'Mean SHAP personal': s[mP].mean() if nP else np.nan, 'Personal minus business': est, 'CI low': lo, 'CI high': hi})
per_r = pd.DataFrame(per_r); per_r.to_csv(f'{OUT_DIR}/table_per_rating_segment.csv', index=False)
print(per_r[per_r.Attribute.isin(KEY)].round(3).to_string())
dist_rows = []
for c in service_cols:
    v = R[c].values
    for tn in ['Business', 'Personal']:
        m = (TT == tn) & ~np.isnan(v)
        dist_rows.append({'Attribute': c, 'Travel': tn, **{f'share {r}': round((v[m] == r).mean(), 3) for r in range(1, 6)}})
pd.DataFrame(dist_rows).to_csv(f'{OUT_DIR}/table_rating_distribution_by_travel.csv', index=False)
print(pd.DataFrame(dist_rows)[lambda d: d.Attribute.isin(KEY)].to_string())

def common_weights(v):
    ok = ~np.isnan(v); vv = v[ok]
    return ({r: (vv == r).mean() for r in [1, 2, 3]}, {r: (vv == r).mean() for r in [4, 5]})

def adjusted_table(S, label, rs, mode, ratings=R, tt=TT):
    """mode: 'reweight' (common rating distribution), 'class' (class-standardised),
    'strata' (class x customer type x distance tertile), 'strata+reweight'."""
    out = []
    for c in service_cols:
        j = fidx[c]; v_all = ratings[c].values; ok = ~np.isnan(v_all); ii0 = np.where(ok)[0]
        wl, wh = common_weights(v_all)
        if mode in ('class',):
            cells = CL
        elif mode in ('strata', 'strata+reweight'):
            cells = np.array([f'{a}|{b}|{d}' for a, b, d in zip(CL, CT, dist_t)])
        else:
            cells = np.array(['all'] * len(v_all))
        cell_names = np.unique(cells[ok]); cell_w = np.array([(cells[ok] == k).mean() for k in cell_names])
        use_w = mode in ('reweight', 'strata+reweight')
        def est(bi):
            v, s, t, cc = v_all[bi], S[bi, j], tt[bi], cells[bi]
            d, w = [], []
            for k, wk in zip(cell_names, cell_w):
                mk = cc == k
                eP = contrast(v, s, mk & (t == 'Personal'), *( (wl, wh) if use_w else (None, None)))
                eB = contrast(v, s, mk & (t == 'Business'), *( (wl, wh) if use_w else (None, None)))
                if not (np.isnan(eP) or np.isnan(eB)): d.append(eP - eB); w.append(wk)
            return np.sum(np.array(w) * np.array(d)) / np.sum(w) if w else np.nan
        e = est(ii0); (lo, hi), _ = boot_diff(lambda b: est(ii0[b]), len(ii0), rs, B=300 if mode.startswith('strata') else B)
        out.append({'Analysis': label, 'Attribute': c, 'Comparison': 'Personal minus business', 'Difference': e, 'CI low': lo, 'CI high': hi})
    return pd.DataFrame(out)
adj_tabs = []
for mode, label in [('reweight', 'Common rating distribution within bins'), ('class', 'Standardised for cabin class'),
                    ('strata', 'Standardised for class, customer type and distance'), ('strata+reweight', 'Strata + common rating distribution')]:
    t0 = time.time(); a = adjusted_table(shap_all, label, np.random.RandomState(SEED), mode); adj_tabs.append(a)
    print(label, f'({time.time() - t0:.0f}s)'); print(a[a.Attribute.isin(KEY)].round(3).to_string())
adj_tabs = pd.concat(adj_tabs); adj_tabs.to_csv(f'{OUT_DIR}/table_adjusted_differences.csv', index=False)
# within-class contrasts and cell sizes (for the text)
wc = []
for c in KEY:
    j = fidx[c]; v = R[c].values
    for cl in ['Business', 'Economy', 'Economy Plus']:
        for tn in ['Business', 'Personal']:
            m = (CL == cl) & (TT == tn)
            wc.append({'Attribute': c, 'Class': cl, 'Travel': tn, 'n': int((m & ~np.isnan(v)).sum()), 'n 4-5': int((m & (v >= 4)).sum()),
                       'Contrast': contrast(v, shap_all[:, j], m)})
pd.DataFrame(wc).to_csv(f'{OUT_DIR}/table_class_stratified_contrasts.csv', index=False); print(pd.DataFrame(wc).round(2).to_string())



In [ ]:
# Cell B4: Interventional TreeSHAP with a background sample - log-odds and probability scales
# Background: 100 passengers drawn from each fold's inner training data. Same 15,000 explained passengers.
cache = f'{OUT_DIR}/shap_interventional.npz'
if os.path.exists(cache):
    zz = np.load(cache); sv_int_raw, sv_int_prob = zz['raw'], zz['prob']
else:
    sv_int_raw, sv_int_prob = [], []
    for s, (tr, te) in enumerate(splits):
        X_in, _, _, _ = train_test_split(X.iloc[tr], y[tr], test_size=0.1, stratify=y[tr], random_state=SEED)
        bg = X_in.sample(100, random_state=SEED)
        Xe = X.iloc[te].iloc[pick_parts[s]]
        t0 = time.time()
        e_raw = shap.TreeExplainer(best_models[s], data=bg, feature_perturbation='interventional', model_output='raw')
        a = np.asarray(e_raw.shap_values(Xe)); a = a[:, :, 1] if a.ndim == 3 else a
        e_prob = shap.TreeExplainer(best_models[s], data=bg, feature_perturbation='interventional', model_output='probability')
        p = np.asarray(e_prob.shap_values(Xe)); p = p[:, :, 1] if p.ndim == 3 else p
        sv_int_raw.append(a.astype('float32')); sv_int_prob.append(p.astype('float32'))
        print(f'Fold {s + 1}: interventional SHAP done ({time.time() - t0:.0f}s)', flush=True)
    sv_int_raw, sv_int_prob = np.vstack(sv_int_raw), np.vstack(sv_int_prob)
    np.savez_compressed(cache, raw=sv_int_raw, prob=sv_int_prob)
sd_int_raw = seg_diff_table(sv_int_raw, 'Interventional, log-odds', np.random.RandomState(SEED))
sd_int_prob = seg_diff_table(sv_int_prob * 100, 'Interventional, probability (percentage points)', np.random.RandomState(SEED))
te_scale = []
for lab, S, mult in [('Path-dependent, log-odds', shap_all, 1), ('Interventional, log-odds', sv_int_raw, 1), ('Interventional, probability (pp)', sv_int_prob, 100)]:
    for c in service_cols:
        j = fidx[c]; v = R[c].values
        row = {'Scale': lab, 'Attribute': c}
        for sn, m in seg_defs[:3]:
            row[sn] = contrast(v, S[:, j] * mult, m & ~np.isnan(v))
        row['Mean |SHAP|'] = np.abs(S[:, j] * mult).mean()
        te_scale.append(row)
te_scale = pd.DataFrame(te_scale); te_scale.to_csv(f'{OUT_DIR}/table_contrasts_by_scale.csv', index=False)
print(te_scale.round(3).to_string())
lv_prob = []
for c in service_cols:
    j = fidx[c]; v = R[c].values
    for sn, m in seg_defs[:3]:
        for r in range(1, 6):
            k = m & (v == r); lv_prob.append({'Attribute': c, 'Segment': sn, 'Rating': r, 'Mean SHAP (pp)': 100 * sv_int_prob[k, j].mean() if k.any() else np.nan,
                                               'Mean SHAP interventional log-odds': sv_int_raw[k, j].mean() if k.any() else np.nan})
pd.DataFrame(lv_prob).to_csv(f'{OUT_DIR}/table_shap_by_rating_interventional.csv', index=False)
print(pd.DataFrame(lv_prob)[lambda d: d.Attribute.isin(KEY)].round(2).to_string())
# ranking agreement across services between scales
from scipy.stats import spearmanr
for lab in ['Interventional, log-odds', 'Interventional, probability (percentage points)']:
    o = sd_int_raw if 'log' in lab else sd_int_prob
    a = sd_main[sd_main.Comparison == 'Personal minus business'].set_index('Attribute')['Difference']
    b = o[o.Comparison == 'Personal minus business'].set_index('Attribute')['Difference']
    print(lab, 'Spearman with main differences:', round(spearmanr(a, b.loc[a.index]).correlation, 3),
          '| sign agreement:', int((np.sign(a) == np.sign(b.loc[a.index])).sum()), 'of', len(a))
# baseline satisfaction context: mean predicted probability by segment among explained passengers
p_oof = st['oof_proba']['LightGBM'][shap_idx]
print('Mean predicted probability: business', p_oof[TT == 'Business'].mean().round(3), '| personal', p_oof[TT == 'Personal'].mean().round(3))



In [ ]:
# Cell B5: TreeSHAP interaction values (path-dependent) for service x travel type, 1,000 passengers per fold
cache = f'{OUT_DIR}/shap_interaction.npz'
N_INT = 1000
if os.path.exists(cache):
    zz = np.load(cache); IV, iv_rows = zz['iv'], zz['rows']
else:
    IV, iv_rows = [], []
    for s in range(N_FOLDS):
        te = splits[s][1]; pk = pick_parts[s][:N_INT]
        t0 = time.time(); iv = np.asarray(shap.TreeExplainer(best_models[s]).shap_interaction_values(X.iloc[te].iloc[pk]))
        if iv.ndim == 4: iv = iv[1]
        IV.append(iv.astype('float32')); iv_rows.append(np.arange(s * 3000, s * 3000 + N_INT))
        print(f'Fold {s + 1}: interaction values done ({time.time() - t0:.0f}s)', flush=True)
    IV, iv_rows = np.concatenate(IV), np.concatenate(iv_rows)
    np.savez_compressed(cache, iv=IV, rows=iv_rows)
jt = fidx['Business travel']
Ri, TTi = R.iloc[iv_rows].reset_index(drop=True), TT[iv_rows]
int_rows = []; rs = np.random.RandomState(SEED)
for c in service_cols:
    j = fidx[c]; v = Ri[c].values; ok = ~np.isnan(v)
    total = IV[:, j, :].sum(axis=1)                 # equals the SHAP value of service j
    main = IV[:, j, j]; inter_tt = IV[:, j, jt]      # half of the symmetric service x travel interaction
    other = total - main - inter_tt
    row = {'Attribute': c, 'n': int(ok.sum()),
           'Share of |SHAP| from travel interaction (%)': 100 * np.abs(inter_tt[ok]).mean() / np.abs(total[ok]).mean()}
    for nm, comp in [('Total', total), ('Main effect', main), ('Travel-type interaction', inter_tt), ('Other interactions', other)]:
        for tn in ['Business', 'Personal']:
            row[f'{nm} contrast, {tn}'] = contrast(v, comp, ok & (TTi == tn))
        row[f'{nm} P-B'] = row[f'{nm} contrast, Personal'] - row[f'{nm} contrast, Business']
    ii = np.where(ok)[0]
    (lo, hi), _ = boot_diff(lambda b: contrast(v[ii[b]], inter_tt[ii[b]], TTi[ii[b]] == 'Personal') - contrast(v[ii[b]], inter_tt[ii[b]], TTi[ii[b]] == 'Business'), len(ii), rs)
    row['Travel-type interaction P-B CI low'], row['Travel-type interaction P-B CI high'] = lo, hi
    # mean interaction term at 4-5 and at 1-3 within segment
    for tn in ['Business', 'Personal']:
        row[f'Mean interaction 4-5, {tn}'] = inter_tt[ok & (TTi == tn) & (v >= 4)].mean()
        row[f'Mean interaction 1-3, {tn}'] = inter_tt[ok & (TTi == tn) & (v <= 3)].mean()
    int_rows.append(row)
int_df = pd.DataFrame(int_rows); int_df.to_csv(f'{OUT_DIR}/table_interaction_decomposition.csv', index=False)
print(int_df.round(3).T.to_string())



In [ ]:
# Cell B6: Model-fitting uncertainty - 5 x 5 repeated cross-validation with refitting
# Each repetition uses a new stratified split (seed 42 + repetition); within each training fold LightGBM is refitted with the
# configuration chosen most often in the main analysis, with early stopping on a fresh inner validation split. SHAP for 3,000 test passengers.
cache = f'{OUT_DIR}/table_repeated_cv_folds.csv'
cfg_mode = int(results_df[results_df.Model == 'LightGBM']['Chosen config'].mode().iloc[0]); cfg = lgb_configs[cfg_mode]
print('Configuration used for repeated CV:', cfg_mode, cfg)
if os.path.exists(cache):
    rcv = pd.read_csv(cache)
else:
    rcv = []
    for rep in range(5):
        sk = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + rep)
        rsr = np.random.RandomState(SEED + rep)
        for s, (tr, te) in enumerate(sk.split(X, y)):
            t0 = time.time()
            Xa, Xv, ya, yv = train_test_split(X.iloc[tr], y[tr], test_size=0.1, stratify=y[tr], random_state=SEED + rep)
            mdl = lgb.LGBMClassifier(n_estimators=2000, subsample_freq=1, random_state=SEED + rep, n_jobs=-1, verbose=-1, **cfg)
            mdl.fit(Xa, ya, eval_set=[(Xv, yv)], eval_metric='binary_logloss', callbacks=[lgb.early_stopping(50, verbose=False)])
            p = mdl.predict_proba(X.iloc[te])[:, 1]
            pk = rsr.choice(len(te), 3000, replace=False); rows_ = te[pk]
            sv = np.asarray(shap.TreeExplainer(mdl).shap_values(X.iloc[rows_])); sv = sv[1] if sv.ndim == 3 and sv.shape[0] == 2 else sv
            if sv.ndim == 3: sv = sv[:, :, 1]
            Rf = df[service_cols].iloc[rows_].reset_index(drop=True); tf = raw['Type of Travel'].values[rows_]
            row = {'Repetition': rep + 1, 'Fold': s + 1, 'Accuracy': accuracy_score(y[te], p >= 0.5), 'F1': f1_score(y[te], p >= 0.5),
                   'ROC-AUC': roc_auc_score(y[te], p), 'Trees': mdl.best_iteration_}
            for c in service_cols:
                v = Rf[c].values; s_ = sv[:, fidx[c]]; ok = ~np.isnan(v)
                eB, eP = contrast(v, s_, ok & (tf == 'Business')), contrast(v, s_, ok & (tf == 'Personal'))
                row[f'{c} | business'] = eB; row[f'{c} | personal'] = eP; row[f'{c} | P-B'] = eP - eB
                row[f'{c} | all'] = contrast(v, s_, ok)
            rcv.append(row); print(f'rep {rep + 1} fold {s + 1}: acc {row["Accuracy"]:.4f}  Wi-Fi P-B {row["In-flight Wifi Service | P-B"]:.2f} ({time.time() - t0:.0f}s)', flush=True)
    rcv = pd.DataFrame(rcv); rcv.to_csv(cache, index=False)
summ = []
for c in service_cols:
    for k in ['all', 'business', 'personal', 'P-B']:
        col = f'{c} | {k}'; per_rep = rcv.groupby('Repetition')[col].mean()
        summ.append({'Attribute': c, 'Quantity': k, 'Mean over 25 folds': rcv[col].mean(), 'SD over 25 folds': rcv[col].std(),
                     'Min fold': rcv[col].min(), 'Max fold': rcv[col].max(), 'Min repetition mean': per_rep.min(), 'Max repetition mean': per_rep.max(),
                     'Folds with P>B': int((rcv[col] > 0).sum()) if k == 'P-B' else np.nan})
summ = pd.DataFrame(summ); summ.to_csv(f'{OUT_DIR}/table_repeated_cv_summary.csv', index=False)
print(summ[summ.Attribute.isin(KEY)].round(3).to_string())
print('Repeated-CV accuracy', rcv['Accuracy'].mean().round(4), '±', rcv['Accuracy'].std().round(4), '| ROC-AUC', rcv['ROC-AUC'].mean().round(4))
print('Number of services whose P-B sign is the same in all 25 folds:',
      sum(int((rcv[f'{c} | P-B'] > 0).all() or (rcv[f'{c} | P-B'] < 0).all()) for c in service_cols))



In [ ]:
# Cell B7: Correlated predictors - grouped attribution and refits that drop one of the two correlated ratings
rho = df[service_cols].corr(method='spearman'); rho.round(2).to_csv(f'{OUT_DIR}/table_spearman_ratings.csv')
print('rho(Wi-Fi, booking) =', round(rho.loc['In-flight Wifi Service', 'Ease of Online Booking'], 3))
# (a) grouped: sum of the Wi-Fi and online-booking attributions, contrast on Wi-Fi rating
grp_rows = []
for lab, S in [('Path-dependent', shap_all), ('Interventional', sv_int_raw)]:
    sg = S[:, fidx['In-flight Wifi Service']] + S[:, fidx['Ease of Online Booking']]
    v = R['In-flight Wifi Service'].values; ok = ~np.isnan(v)
    eB, eP = contrast(v, sg, ok & (TT == 'Business')), contrast(v, sg, ok & (TT == 'Personal'))
    sw = S[:, fidx['In-flight Wifi Service']]
    grp_rows.append({'Method': lab, 'Wi-Fi only, business': contrast(v, sw, ok & (TT == 'Business')), 'Wi-Fi only, personal': contrast(v, sw, ok & (TT == 'Personal')),
                     'Wi-Fi + booking, business': eB, 'Wi-Fi + booking, personal': eP, 'Wi-Fi + booking P-B': eP - eB})
# (b) refits without online booking (and without Wi-Fi) - same fold configurations, out-of-fold SHAP on the same passengers
drop_res = {}
for drop in ['Ease of Online Booking', 'In-flight Wifi Service']:
    cache_d = f'{OUT_DIR}/shap_drop_{drop.replace(" ", "_")}.npy'
    keep = [c for c in feature_names if c != drop]
    if os.path.exists(cache_d):
        Sd = np.load(cache_d)
    else:
        parts = []
        for s, (tr, te) in enumerate(splits):
            cfg_s = lgb_configs[int(results_df[(results_df.Model == 'LightGBM') & (results_df.Fold == s + 1)]['Chosen config'].iloc[0])]
            Xa, Xv, ya, yv = train_test_split(X.iloc[tr][keep], y[tr], test_size=0.1, stratify=y[tr], random_state=SEED)
            mdl = lgb.LGBMClassifier(n_estimators=2000, subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1, **cfg_s)
            mdl.fit(Xa, ya, eval_set=[(Xv, yv)], eval_metric='binary_logloss', callbacks=[lgb.early_stopping(50, verbose=False)])
            sv = np.asarray(shap.TreeExplainer(mdl).shap_values(X[keep].iloc[te].iloc[pick_parts[s]]))
            if sv.ndim == 3: sv = sv[:, :, 1]
            parts.append(sv.astype('float32')); print(f'drop {drop} fold {s + 1}: acc {accuracy_score(y[te], mdl.predict_proba(X[keep].iloc[te])[:, 1] >= 0.5):.4f}', flush=True)
        Sd = np.vstack(parts); np.save(cache_d, Sd)
    drop_res[drop] = (Sd, keep)
for drop, (Sd, keep) in drop_res.items():
    other = 'In-flight Wifi Service' if drop == 'Ease of Online Booking' else 'Ease of Online Booking'
    v = R[other].values; ok = ~np.isnan(v); sj = Sd[:, keep.index(other)]
    eB, eP = contrast(v, sj, ok & (TT == 'Business')), contrast(v, sj, ok & (TT == 'Personal'))
    grp_rows.append({'Method': f'Refit without {drop}: {other}', f'Wi-Fi only, business' if other.startswith('In-flight') else 'Booking only, business': eB,
                     f'Wi-Fi only, personal' if other.startswith('In-flight') else 'Booking only, personal': eP, 'Refit P-B': eP - eB,
                     'Mean |SHAP| refit': np.abs(sj).mean(), 'Mean |SHAP| main': np.abs(shap_all[:, fidx[other]]).mean()})
grp = pd.DataFrame(grp_rows); grp.to_csv(f'{OUT_DIR}/table_correlated_predictors.csv', index=False); print(grp.round(3).T.to_string())
sd_drop = seg_diff_table(np.insert(drop_res['Ease of Online Booking'][0], fidx['Ease of Online Booking'], 0, axis=1), 'Refit without online booking', np.random.RandomState(SEED))
print(sd_drop[sd_drop.Attribute.isin(KEY) & (sd_drop.Comparison == 'Personal minus business')].round(3).to_string())



In [ ]:
# Cell B8: Sensitivity checks carried over - zero ratings kept as scores (prediction) and refit without zero-raters (attributions)
Xz = X.copy(); Xz[service_cols] = raw[service_cols].values.astype('float32')
rows = []
for s, (tr, te) in enumerate(splits):
    cfg_s = lgb_configs[int(results_df[(results_df.Model == 'LightGBM') & (results_df.Fold == s + 1)]['Chosen config'].iloc[0])]
    Xa, Xv, ya, yv = train_test_split(Xz.iloc[tr], y[tr], test_size=0.1, stratify=y[tr], random_state=SEED)
    mdl = lgb.LGBMClassifier(n_estimators=2000, subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1, **cfg_s)
    mdl.fit(Xa, ya, eval_set=[(Xv, yv)], eval_metric='binary_logloss', callbacks=[lgb.early_stopping(50, verbose=False)])
    p = mdl.predict_proba(Xz.iloc[te])[:, 1]; rows.append({'F1': f1_score(y[te], p >= 0.5), 'ROC-AUC': roc_auc_score(y[te], p)})
zs = pd.DataFrame(rows).mean(); main_l = results_df[results_df.Model == 'LightGBM'][['F1', 'ROC-AUC']].mean()
pd.DataFrame({'0 as missing': main_l, '0 as score': zs}).to_csv(f'{OUT_DIR}/table_zero_sensitivity.csv'); print(pd.DataFrame({'0 as missing': main_l, '0 as score': zs}).round(4))
nzm = ~(raw[service_cols] == 0).any(axis=1).values
X_nz, y_nz = X[nzm].reset_index(drop=True), y[nzm]; R_nz = df.loc[nzm, service_cols].reset_index(drop=True); tt_nz = raw.loc[nzm, 'Type of Travel'].values
rs_nz = np.random.RandomState(SEED); Sn, In = [], []
for s, (tr, te) in enumerate(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_nz, y_nz)):
    cfg_s = lgb_configs[int(results_df[(results_df.Model == 'LightGBM') & (results_df.Fold == s + 1)]['Chosen config'].iloc[0])]
    Xa, Xv, ya, yv = train_test_split(X_nz.iloc[tr], y_nz[tr], test_size=0.1, stratify=y_nz[tr], random_state=SEED)
    mdl = lgb.LGBMClassifier(n_estimators=2000, subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1, **cfg_s)
    mdl.fit(Xa, ya, eval_set=[(Xv, yv)], eval_metric='binary_logloss', callbacks=[lgb.early_stopping(50, verbose=False)])
    pk = rs_nz.choice(len(te), 3000, replace=False)
    sv = np.asarray(shap.TreeExplainer(mdl).shap_values(X_nz.iloc[te[pk]])); sv = sv[:, :, 1] if sv.ndim == 3 else sv
    Sn.append(sv); In.append(te[pk])
Sn, In = np.vstack(Sn), np.concatenate(In)
sd_nz = seg_diff_table(Sn, 'Refit without passengers with any zero rating', np.random.RandomState(SEED),
                       ratings=R_nz.iloc[In].reset_index(drop=True), tt=tt_nz[In], ct=raw.loc[nzm, 'Customer Type'].values[In])
sd_nz.to_csv(f'{OUT_DIR}/table_sensitivity_no_zero_ratings.csv', index=False)
print(sd_nz[sd_nz.Attribute.isin(KEY)].round(3).to_string())
v = R_nz.iloc[In]['In-flight Wifi Service'].values
print('No-zero refit Wi-Fi mean SHAP by rating:', [round(Sn[v == r, fidx['In-flight Wifi Service']].mean(), 2) for r in range(1, 6)])
print('No-zero refit: services whose overall contrast keeps its sign:',
      sum(np.sign(contrast(R_nz.iloc[In][c].values, Sn[:, fidx[c]], ~np.isnan(R_nz.iloc[In][c].values))) ==
          np.sign(te_tab[(te_tab.Segment == 'All') & (te_tab.Attribute == c)]['Contrast'].iloc[0]) for c in service_cols))



In [ ]:
# Cell B9: Logistic-regression C profile (validation ROC-AUC for every C in every fold)
lr_grid = [1e-3, 1e-2, 1e-1, 1, 10, 100, 1000]; prof = []
for s, (tr, te) in enumerate(splits):
    X_in, X_val, y_in, y_val = train_test_split(X.iloc[tr], y[tr], test_size=0.1, stratify=y[tr], random_state=SEED)
    med = X_in[['Age', 'Flight Distance', 'Departure Delay', 'Arrival Delay'] + service_cols].median()
    def prep(p):
        p = p.copy()
        for c in service_cols + ['Arrival Delay']: p[c + ' (missing)'] = p[c].isna().astype('float32')
        return p.fillna(med)
    a, b_ = prep(X_in), prep(X_val); sc = StandardScaler().fit(a)
    for C in lr_grid:
        m = LogisticRegression(C=C, max_iter=3000, random_state=SEED).fit(sc.transform(a), y_in)
        prof.append({'Fold': s + 1, 'C': C, 'Validation ROC-AUC': roc_auc_score(y_val, m.predict_proba(sc.transform(b_))[:, 1])})
prof = pd.DataFrame(prof); prof.to_csv(f'{OUT_DIR}/table_lr_c_profile.csv', index=False)
print(prof.pivot(index='C', columns='Fold', values='Validation ROC-AUC').round(5).to_string())



In [ ]:
# Cell B10: Penalty-reward check (supplementary; reversed = ratings 4-5 below rating 3)
pr = []; rs_pr = np.random.RandomState(SEED)
def _ia(s_, lo_, mid, hi_):
    pen, rew = max(0.0, -(s_[lo_].mean() - s_[mid].mean())), max(0.0, s_[hi_].mean() - s_[mid].mean())
    return (rew - pen) / (rew + pen) if rew + pen > 0 else np.nan
for sn, m in seg_defs:
    for c in service_cols:
        j = fidx[c]; ii = np.where(m & R[c].notna().values)[0]; v = R[c].values[ii]; s_ = shap_all[ii, j]
        lo_, mid, hi_ = (v <= 2), (v == 3), (v >= 4)
        P_raw, R_raw = s_[lo_].mean() - s_[mid].mean(), s_[hi_].mean() - s_[mid].mean()
        pen, rew = max(0, -P_raw), max(0, R_raw)
        bs = [_ia(s_[b], lo_[b], mid[b], hi_[b]) for b in (rs_pr.randint(0, len(ii), len(ii)) for _ in range(300))]
        ci_lo, ci_hi = np.nanpercentile(bs, [2.5, 97.5])
        if R_raw < -0.1 or (R_raw < 0 and P_raw > 0): kind = 'Reversed'
        elif pen + rew < 0.1: kind = 'Weak'
        elif ci_hi < -0.2: kind = 'Basic'
        elif ci_lo > 0.2: kind = 'Excitement'
        else: kind = 'Performance'
        pr.append({'Segment': sn, 'Attribute': c, 'Penalty (1-2 vs 3)': P_raw, 'Reward (4-5 vs 3)': R_raw,
                   'IA CI low': ci_lo, 'IA CI high': ci_hi, 'Type': kind})
pr = pd.DataFrame(pr); pr.to_csv(f'{OUT_DIR}/table_penalty_reward_supplement.csv', index=False)
print(pr.pivot(index='Attribute', columns='Segment', values='Type').to_string())
print('done')


## 3. Robustness table (Table VI)

In [ ]:
# Cell B11: Robustness table (Table VI) - segment differences under every check, with 95% intervals
import pickle, numpy as np, pandas as pd
zz = np.load(f'{OUT_DIR}/shap_interventional.npz'); sv_int_raw, sv_int_prob = zz['raw'], zz['prob']
sd_main = pd.read_csv(f'{OUT_DIR}/table_segment_differences.csv')
sd_main['Comparison'] = sd_main['Comparison'].str.replace('Business', 'business').str.replace('Returning', 'returning')
sd_int_raw = seg_diff_table(sv_int_raw, 'Interventional SHAP, log-odds', np.random.RandomState(SEED))
sd_int_prob = seg_diff_table(sv_int_prob * 100, 'Interventional SHAP, probability scale (percentage points)', np.random.RandomState(SEED))
pd.concat([sd_int_raw, sd_int_prob]).to_csv(f'{OUT_DIR}/table_segment_differences_interventional.csv', index=False)
from scipy.stats import spearmanr
a = sd_main[sd_main.Comparison == 'Personal minus business'].set_index('Attribute')['Difference']
for o in [sd_int_raw, sd_int_prob]:
    b = o[o.Comparison == 'Personal minus business'].set_index('Attribute')['Difference'].loc[a.index]
    print(o.Analysis.iloc[0], '| Spearman with main:', round(spearmanr(a, b).correlation, 3), '| same sign:', int((np.sign(a) == np.sign(b)).sum()), 'of 14')
    print(pd.DataFrame({'main': a, 'alt': b}).round(3).to_string())
adj = pd.read_csv(f'{OUT_DIR}/table_adjusted_differences.csv')
nz = pd.read_csv(f'{OUT_DIR}/table_sensitivity_no_zero_ratings.csv'); nz['Comparison'] = nz['Comparison'].str.replace('Business', 'business').str.replace('Returning', 'returning')
rcv = pd.read_csv(f'{OUT_DIR}/table_repeated_cv_folds.csv')
iv = pd.read_csv(f'{OUT_DIR}/table_interaction_decomposition.csv').set_index('Attribute')
rows = []
def get(t, label=None, att=None):
    d = t[(t.Comparison == 'Personal minus business') & (t.Attribute == att)]
    if label is not None: d = d[d.Analysis == label]
    r = d.iloc[0]; return r['Difference'], r['CI low'], r['CI high']
for lab, t, l2 in [('Main analysis (path-dependent, log-odds)', sd_main, None),
                   ('Interventional SHAP, log-odds', sd_int_raw, None),
                   ('Interventional SHAP, probability (percentage points)', sd_int_prob, None),
                   ('Common rating distribution within the 1–3 and 4–5 bins', adj, 'Common rating distribution within bins'),
                   ('Standardised for cabin class', adj, 'Standardised for cabin class'),
                   ('Standardised for class, customer type and flight distance', adj, 'Standardised for class, customer type and distance'),
                   ('Refitted without passengers with any zero rating', nz, None)]:
    row = {'Analysis': lab}
    for att in KEY:
        row[att] = get(t, l2, att)
    rows.append(row)
row = {'Analysis': 'Service × travel-type interaction term only (5,000 passengers)'}
for att in KEY:
    row[att] = (iv.loc[att, 'Travel-type interaction P-B'], iv.loc[att, 'Travel-type interaction P-B CI low'], iv.loc[att, 'Travel-type interaction P-B CI high'])
rows.append(row)
row = {'Analysis': '5 × 5 repeated CV with refitting: mean (range over 25 folds)'}
for att in KEY:
    c = rcv[f'{att} | P-B']; row[att] = (c.mean(), c.min(), c.max())
rows.append(row)
if True:
    try:
        Sd = np.load(f'{OUT_DIR}/shap_drop_Ease_of_Online_Booking.npy')
        Sfull = np.insert(Sd, fidx['Ease of Online Booking'], 0, axis=1)
        t = seg_diff_table(Sfull, 'Refit without online booking', np.random.RandomState(SEED))
        row = {'Analysis': 'Refitted without ease of online booking'}
        row['In-flight Wifi Service'] = get(t, None, 'In-flight Wifi Service'); row['Online Boarding'] = get(t, None, 'Online Boarding')
        rows.append(row); t.to_csv(f'{OUT_DIR}/table_segment_differences_no_booking.csv', index=False)
    except FileNotFoundError:
        pass
t6 = pd.DataFrame(rows)
for att in KEY:
    t6[att] = t6[att].apply(lambda x: f'{x[0]:.2f} ({x[1]:.2f} to {x[2]:.2f})'.replace('-', '−'))
t6.to_csv(f'{OUT_DIR}/table6_robustness.csv', index=False); print(t6.to_string())


## 4. Figures 1-3

In [ ]:
# Cell C1: Figures 1-3 (white background, 600 dpi)
import pickle, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import shap
OUT_DIR = 'results'; FIG_DIR = 'figures'
import os; os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'savefig.facecolor': 'white', 'figure.facecolor': 'white', 'axes.facecolor': 'white'})
st = pickle.load(open(f'{OUT_DIR}/state.pkl', 'rb'))
feature_names, service_cols, shap_idx = st['feature_names'], st['service_cols'], st['shap_idx']
raw = pd.read_csv('airline_passenger_satisfaction.csv')
df = raw.copy(); df[service_cols] = df[service_cols].replace(0, np.nan)
X = df[['Age', 'Flight Distance', 'Departure Delay', 'Arrival Delay'] + service_cols].copy()
X['Male'] = (df['Gender'] == 'Male').astype(float); X['First-time customer'] = (df['Customer Type'] == 'First-time').astype(float)
X['Business travel'] = (df['Type of Travel'] == 'Business').astype(float)
for cl in ['Business', 'Economy', 'Economy Plus']: X[f'Class {cl}'] = (df['Class'] == cl).astype(float)
S = np.load(f'{OUT_DIR}/shap_values_oof.npy')
short = {'Departure and Arrival Time Convenience': 'Time convenience', 'Ease of Online Booking': 'Online booking',
         'Check-in Service': 'Check-in', 'Online Boarding': 'Online boarding', 'Gate Location': 'Gate location',
         'On-board Service': 'On-board service', 'Seat Comfort': 'Seat comfort', 'Leg Room Service': 'Leg room',
         'Cleanliness': 'Cleanliness', 'Food and Drink': 'Food and drink', 'In-flight Service': 'In-flight service',
         'In-flight Wifi Service': 'In-flight Wi-Fi', 'In-flight Entertainment': 'Entertainment', 'Baggage Handling': 'Baggage handling',
         'Arrival Delay': 'Arrival delay', 'Departure Delay': 'Departure delay', 'Flight Distance': 'Flight distance',
         'Class Business': 'Business class', 'Class Economy': 'Economy class', 'Class Economy Plus': 'Economy plus class'}

# Fig. 1: beeswarm, class dummies kept separate (3 levels), binaries now single columns
names = [short.get(f, f) for f in feature_names]
plt.figure()
shap.summary_plot(S, features=X.iloc[shap_idx].values, feature_names=names, max_display=15, show=False, plot_size=(5.2, 5.6))
fig = plt.gcf(); fig.patch.set_facecolor('white')
ax = plt.gca(); ax.set_xlabel('SHAP value (log-odds of satisfaction)', fontsize=10); ax.tick_params(labelsize=9)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig1_shap_summary.png', dpi=600, bbox_inches='tight', facecolor='white'); plt.close()

# Fig. 2: heatmaps of mean SHAP by rating with passenger counts, one colour scale for all panels
lv = pd.read_csv(f'{OUT_DIR}/table_shap_by_rating.csv')
te = pd.read_csv(f'{OUT_DIR}/table_top_rating_effects.csv')
order = te[te.Segment == 'All'].sort_values('Contrast', ascending=False)['Attribute'].tolist()
fig, axes = plt.subplots(1, 3, figsize=(7.16, 5.6), sharey=True)
norm = TwoSlopeNorm(vmin=-2.5, vcenter=0, vmax=2.5)
for ax, sn, title in zip(axes, ['All', 'Business travel', 'Personal travel'], ['All passengers', 'Business travel', 'Personal travel']):
    M = lv[lv.Segment == sn].pivot(index='Attribute', columns='Rating', values='Mean SHAP').loc[order]
    N = lv[lv.Segment == sn].pivot(index='Attribute', columns='Rating', values='Passengers').loc[order]
    im = ax.imshow(M.values, cmap='RdBu_r', norm=norm, aspect='auto')
    for i in range(M.shape[0]):
        for k in range(5):
            val, n = M.values[i, k], int(N.values[i, k])
            col = 'white' if abs(val) > 1.6 else 'black'
            ax.text(k, i - 0.14, f'{val:.2f}'.replace('-', '−'), ha='center', va='center', fontsize=5.6, color=col, fontweight='bold')
            ax.text(k, i + 0.24, f'n={n:,}', ha='center', va='center', fontsize=4.3, color=col)
    ax.set_xticks(range(5)); ax.set_xticklabels([1, 2, 3, 4, 5], fontsize=7); ax.set_title(title, fontsize=8)
    ax.set_xlabel('Rating', fontsize=7)
    ax.set_xticks(np.arange(-.5, 5), minor=True); ax.set_yticks(np.arange(-.5, len(order)), minor=True)
    ax.grid(which='minor', color='white', lw=0.8); ax.tick_params(which='minor', length=0)
axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels([short[c] for c in order], fontsize=7)
axes[0].set_ylabel('Service attribute', fontsize=7)
cb = fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.035, pad=0.09, extend='both')
cb.set_label('Mean SHAP value (log-odds of satisfaction); colours truncated at ±2.5, printed values exact', fontsize=6.5); cb.ax.tick_params(labelsize=6)
plt.savefig(f'{FIG_DIR}/fig2_rating_profiles.png', dpi=600, bbox_inches='tight', facecolor='white'); plt.close()

# Fig. 3: top-rating contrast with 95% intervals and data labels
fig, axes = plt.subplots(1, 2, figsize=(7.16, 4.6), sharey=True)
order3 = order[::-1]; ypos = np.arange(len(order3)); h = 0.38
for ax, pairs, title in [(axes[0], [('Business travel', '#2a6fdb'), ('Personal travel', '#e07b00')], 'By type of travel'),
                         (axes[1], [('Returning customers', '#2a9d8f'), ('First-time customers', '#b5179e')], 'By customer type')]:
    for k, (sn, col) in enumerate(pairs):
        d = te[te.Segment == sn].set_index('Attribute').loc[order3]
        yy = ypos + (k - 0.5) * h
        ax.barh(yy, d['Contrast'], h, color=col, label=sn.replace(' travel', ' travellers') if 'travel' in sn else sn,
                xerr=[d['Contrast'] - d['CI low'], d['CI high'] - d['Contrast']], error_kw={'lw': 0.6, 'capsize': 1.5})
        for y_, c_, hi_, lo_ in zip(yy, d['Contrast'], d['CI high'], d['CI low']):
            xlab = hi_ + 0.08 if c_ >= 0 else lo_ - 0.08
            ax.text(xlab, y_, f'{c_:.2f}'.replace('-', '−'), va='center', ha='left' if c_ >= 0 else 'right', fontsize=5.2)
    ax.axvline(0, color='k', lw=0.6); ax.set_title(title, fontsize=8); ax.legend(fontsize=6, loc='lower right', frameon=False)
    ax.set_xlabel('Top-rating contrast: mean SHAP at 4–5 minus 1–3\n(log-odds of satisfaction)', fontsize=7)
    ax.tick_params(labelsize=7); ax.set_xlim(-1.0, 6.9)
axes[0].set_yticks(ypos); axes[0].set_yticklabels([short[c] for c in order3], fontsize=7)
axes[0].set_ylabel('Service attribute', fontsize=7)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/fig3_top_rating_contrast.png', dpi=600, bbox_inches='tight', facecolor='white'); plt.close()
print('figures done')


## 5. External plausibility check (Skytrax reviews)

In [ ]:
# Cell D1: Plausibility check on an independent source (Skytrax user reviews, scraped 2 Aug 2015; quankiquanki/skytrax-reviews-dataset)
import numpy as np, pandas as pd
SKYTRAX_PATH = 'skytrax_airline.csv'   # data/airline.csv from github.com/quankiquanki/skytrax-reviews-dataset
d = pd.read_csv(SKYTRAX_PATH)
d = d.dropna(subset=['overall_rating', 'wifi_connectivity_rating'])
d['Outcome'] = pd.cut(d['overall_rating'], [0, 4, 6, 10], labels=['Dissatisfied (1-4)', 'Neutral (5-6)', 'Satisfied (7-10)'])
d['Purpose'] = np.where(d['type_traveller'] == 'Business', 'Business', np.where(d['type_traveller'].notna(), 'Leisure', 'Unknown'))
print('Reviews with overall and Wi-Fi ratings:', len(d), d['Purpose'].value_counts().to_dict(), d['airline_name'].nunique(), 'airlines')
tab = pd.crosstab(d['wifi_connectivity_rating'], d['Outcome'], normalize='index').mul(100).round(1)
tab['n'] = d['wifi_connectivity_rating'].value_counts().sort_index(); print(tab)
tab.to_csv('results/table_skytrax_wifi_outcome.csv')
for p in ['Business', 'Leisure']:
    g = d[d.Purpose == p]
    t = g.groupby(pd.cut(g['wifi_connectivity_rating'], [0, 3, 5], labels=['1-3', '4-5']))['Outcome'].apply(lambda s: round(100 * (s == 'Satisfied (7-10)').mean(), 1))
    n = g.groupby(pd.cut(g['wifi_connectivity_rating'], [0, 3, 5], labels=['1-3', '4-5'])).size()
    print(p, dict(t), dict(n))
    pd.DataFrame({'% satisfied': t, 'n': n}).to_csv(f'results/table_skytrax_{p.lower()}.csv')
print('Spearman(wifi, overall):', round(d[['wifi_connectivity_rating', 'overall_rating']].corr('spearman').iloc[0, 1], 3))
print('Spearman(seat, overall):', round(d[['seat_comfort_rating', 'overall_rating']].dropna().corr('spearman').iloc[0, 1], 3))
print('date range', d['date'].min(), d['date'].max())
